# Sistema de Detección de Intrusiones basado en Inteligencia Artificial
## Fortalecimiento de la Ciberseguridad en Redes TCP/IP

**Autor:** Nicolás Martínez Pineda
**Asignatura:** Fundamentos de Redes y Comunicación — Universidad Distrital Francisco José de Caldas
**Docente:** Octavio José Salcedo Parra

Este cuaderno documenta el desarrollo experimental del proyecto conforme a la metodología basada en CRISP-DM
(Chapman et al., 2000): comprensión del problema, preparación de datos, modelado, evaluación y despliegue,
integrando el módulo de identificación de huella digital de dispositivos (IEEE 802.3) con los modelos
de clasificación **Random Forest** (Breiman, 2001) y **XGBoost** (Chen & Guestrin, 2016).

**Estructura del cuaderno:**
1. Configuración del entorno
2. Fase 1 — Comprensión del problema (variables y vectores de ataque por capa)
3. Fase 2 — Captura y extracción de huella digital (parámetros IEEE 802.3)
4. Fase 3 — Preparación del conjunto de datos de tráfico (legítimo / malicioso)
5. Fase 4 — Modelado: Random Forest vs. XGBoost
6. Fase 5 — Evaluación comparativa (precisión, sensibilidad, F1-score, tasa de falsos positivos)
7. Fase 6 — Simulación de ataque a la huella digital y evaluación de la defensa
8. Fase 7 — Diagnóstico, mitigación y pronóstico


In [ ]:
# 1. Configuración del entorno
# Ejecutar solo si hace falta instalar dependencias:
# !pip install scikit-learn xgboost pandas numpy matplotlib scapy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Fase 1 — Comprensión del problema

Vectores de ataque considerados por capa (TCP/IP e ISO/OSI), según la caracterización documental del proyecto:

| Capa | Vector de ataque | Relevancia para la huella digital |
|---|---|---|
| Enlace | ARP spoofing | Altera la asociación IP–MAC, afecta directamente la huella IEEE 802.3 |
| Red | Suplantación de IP, DDoS | Afecta el volumen y origen del tráfico |
| Transporte | SYN flood | Afecta patrones de establecimiento de conexión |
| Aplicación | DNS spoofing, phishing | Afecta el comportamiento de la capa superior, no la huella física |

Estas categorías guían las variables que se extraen en la Fase 2 y las que se usan como entrada de los modelos en la Fase 4.


In [ ]:
# 2. Extracción de huella digital de dispositivos (IEEE 802.3)
#
# Esta función define el esquema de captura en tiempo real con Scapy.
# Requiere permisos de captura de red (sudo / administrador) y una interfaz válida.
# Se deja como plantilla: en el entorno de laboratorio del curso se reemplaza `iface`
# por la interfaz real y se ejecuta fuera de este cuaderno (script aparte) o en una
# celda con permisos elevados.

from collections import defaultdict

def extraer_huella_digital(paquete):
    """Extrae parámetros de huella digital IEEE 802.3 de un paquete Ethernet.

    Parámetros extraídos:
      - mac_origen: dirección MAC de origen
      - oui: identificador de fabricante (primeros 3 octetos de la MAC)
      - longitud_trama: tamaño de la trama Ethernet
      - ethertype: tipo de protocolo encapsulado
    """
    try:
        mac_origen = paquete.src
        oui = mac_origen.upper().replace(':', '')[:6]
        longitud_trama = len(paquete)
        ethertype = paquete.type if hasattr(paquete, 'type') else None
        return {
            'mac_origen': mac_origen,
            'oui': oui,
            'longitud_trama': longitud_trama,
            'ethertype': ethertype,
        }
    except AttributeError:
        return None

# Ejemplo de uso con captura real (comentado — requiere permisos y Scapy):
#
# from scapy.all import sniff
# capturas = []
# def _callback(pkt):
#     huella = extraer_huella_digital(pkt)
#     if huella:
#         capturas.append(huella)
# sniff(iface="eth0", prn=_callback, count=500, timeout=60)
# df_huellas = pd.DataFrame(capturas)

print("Función de extracción de huella digital definida. Pendiente: captura en el laboratorio de la asignatura.")


## Fase 3 — Preparación del conjunto de datos

Como conjunto de datos de referencia se propone **CICIDS2017/2018** (Sharafaldin et al., 2018), por su cobertura de
tráfico legítimo y malicioso etiquetado (DDoS, port scan, brute force, entre otros) y su uso extendido en la
literatura de IDS basados en aprendizaje automático (Ahmad et al., 2021; Pinto et al., 2023).

**Nota de trazabilidad:** mientras se gestiona el acceso y la descarga del dataset completo (varios GB, requiere
almacenamiento externo), esta sección genera un conjunto de datos sintético con la misma estructura de
características (huella digital + variables de tráfico), para dejar el pipeline de modelado completamente
funcional y listo para sustituir por los datos reales sin cambiar el resto del cuaderno.


In [ ]:
# 3. Conjunto de datos — carga real o generación sintética de respaldo

RUTA_DATASET = "data/cicids2017_procesado.csv"  # Ruta esperada del dataset real preprocesado

def cargar_dataset(ruta=RUTA_DATASET, n_sintetico=5000):
    try:
        df = pd.read_csv(ruta)
        print(f"Dataset real cargado desde {ruta}: {df.shape}")
        return df
    except FileNotFoundError:
        print("Dataset real no encontrado todavía. Generando datos sintéticos de respaldo "
              "para validar el pipeline (reemplazar en cuanto el dataset esté disponible).")
        rng = np.random.default_rng(RANDOM_STATE)
        n = n_sintetico
        df = pd.DataFrame({
            'longitud_trama': rng.integers(60, 1518, n),
            'tiempo_entre_tramas_ms': rng.exponential(5, n),
            'num_conexiones_por_min': rng.poisson(20, n),
            'tasa_syn': rng.uniform(0, 1, n),
            'entropia_puertos_destino': rng.uniform(0, 5, n),
            'oui_frecuente': rng.integers(0, 50, n),  # fabricante codificado
        })
        # Etiqueta: malicioso si el tráfico muestra patrones anómalos combinados
        score = (
            (df['tasa_syn'] > 0.7).astype(int) +
            (df['num_conexiones_por_min'] > 35).astype(int) +
            (df['entropia_puertos_destino'] > 3.5).astype(int)
        )
        df['etiqueta'] = (score >= 2).astype(int)  # 1 = malicioso, 0 = legítimo
        return df

df = cargar_dataset()
df.head()


In [ ]:
# Partición de variables predictoras y variable objetivo
X = df.drop(columns=['etiqueta'])
y = df['etiqueta']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc = scaler.transform(X_test)

print(f"Entrenamiento: {X_train.shape}, Prueba: {X_test.shape}")
print(f"Distribución de clases (train): {y_train.value_counts(normalize=True).to_dict()}")


## Fase 4 — Modelado: Random Forest vs. XGBoost

Se entrenan los dos modelos definidos en los objetivos específicos del proyecto para comparar su desempeño
en la clasificación del tráfico como legítimo o malicioso.


In [ ]:
# 4. Entrenamiento de modelos

modelo_rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1
)
modelo_rf.fit(X_train_esc, y_train)

modelo_xgb = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
)
modelo_xgb.fit(X_train_esc, y_train)

print("Modelos entrenados: Random Forest y XGBoost.")


## Fase 5 — Evaluación comparativa

Métricas: exactitud (accuracy), precisión, sensibilidad (recall), F1-score y tasa de falsos positivos (FPR),
conforme a lo definido en el tercer objetivo específico del proyecto.


In [ ]:
# 5. Evaluación comparativa

def evaluar_modelo(modelo, X_test, y_test, nombre):
    y_pred = modelo.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return {
        'modelo': nombre,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        'tasa_falsos_positivos': fpr,
    }

resultados = pd.DataFrame([
    evaluar_modelo(modelo_rf, X_test_esc, y_test, 'Random Forest'),
    evaluar_modelo(modelo_xgb, X_test_esc, y_test, 'XGBoost'),
])
resultados.set_index('modelo').round(4)


In [ ]:
# Visualización comparativa
resultados_plot = resultados.set_index('modelo')[['accuracy', 'precision', 'recall', 'f1_score']]
resultados_plot.plot(kind='bar', figsize=(8, 5), rot=0)
plt.title('Comparación de desempeño: Random Forest vs. XGBoost')
plt.ylabel('Valor de la métrica')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Fase 6 — Simulación de ataque a la huella digital y evaluación de la defensa

Se simula un ataque de suplantación (spoofing) dirigido a alterar la huella digital de un dispositivo
(p. ej. modificando la MAC de origen o el patrón de tramas), y se evalúa si el modelo entrenado logra
seguir clasificando correctamente el tráfico como malicioso — es decir, la capacidad de defensa del prototipo.


In [ ]:
# 6. Simulación de ataque dirigido a la huella digital

def simular_ataque_spoofing(X_legitimo, rng=None):
    """Genera una copia perturbada del tráfico legítimo simulando suplantación
    de huella digital (spoofing de MAC / alteración de patrones de trama)."""
    rng = rng or np.random.default_rng(RANDOM_STATE + 1)
    X_ataque = X_legitimo.copy()
    X_ataque['oui_frecuente'] = rng.integers(0, 50, len(X_ataque))  # MAC/fabricante falsificado
    X_ataque['longitud_trama'] = X_ataque['longitud_trama'] * rng.uniform(0.9, 1.1, len(X_ataque))
    X_ataque['tasa_syn'] = np.clip(X_ataque['tasa_syn'] + rng.uniform(0.3, 0.6, len(X_ataque)), 0, 1)
    return X_ataque

# Tomar tráfico legítimo del set de prueba original como base del ataque simulado
idx_legitimo = y_test[y_test == 0].index
X_legitimo_test = X_test.loc[idx_legitimo]

X_ataque_simulado = simular_ataque_spoofing(X_legitimo_test)
X_ataque_esc = scaler.transform(X_ataque_simulado)
y_ataque_esperado = pd.Series(1, index=X_ataque_simulado.index)  # se espera que ahora se detecte como malicioso

deteccion_rf = evaluar_modelo(modelo_rf, X_ataque_esc, y_ataque_esperado, 'Random Forest (bajo ataque)')
deteccion_xgb = evaluar_modelo(modelo_xgb, X_ataque_esc, y_ataque_esperado, 'XGBoost (bajo ataque)')

pd.DataFrame([deteccion_rf, deteccion_xgb]).set_index('modelo')[['recall']].rename(
    columns={'recall': 'tasa_de_deteccion_del_ataque'}
).round(4)


## Fase 7 — Diagnóstico, mitigación y pronóstico

- **Diagnóstico:** completar con el análisis comparativo de la Fase 5 una vez sustituido el dataset sintético
  por el dataset real (CICIDS2017/2018) y por la captura propia de huella digital.
- **Mitigación:** definir reglas de respuesta ante tráfico clasificado como anómalo (p. ej. cuarentena del
  dispositivo, alerta, bloqueo temporal por MAC/OUI).
- **Pronóstico:** con los resultados de la Fase 6, proyectar la tendencia de comportamiento anómalo en la red
  (por ejemplo, mediante una serie de tiempo de la tasa de detección de ataques a lo largo de varias simulaciones).

**Pendientes de trazabilidad (próximos cortes del curso):**
1. Reemplazar el dataset sintético por CICIDS2017/2018 real.
2. Ejecutar la captura de huella digital en el laboratorio (Fase 2) e integrarla como variable adicional.
3. Ajuste de hiperparámetros (`GridSearchCV` / `RandomizedSearchCV`) para ambos modelos.
4. Documentar el diagnóstico, la mitigación y el pronóstico con los resultados reales para el entregable final.
